# Silver Layer - Generic Metadata-Driven Processing

Generic Bronze-to-Silver notebook driven by `TABLE_ID` metadata.


In [ ]:
%run ../helpers/NB_catalog_helpers

In [ ]:
%run ../helpers/NB_schema_drift_helpers

In [ ]:
%run ../helpers/NB_schema_contracts

In [ ]:
%run ../helpers/NB_silver_metadata

In [ ]:
%run ../helpers/NB_key_management_helpers

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import coalesce, col, expr, from_json, get_json_object, lit, row_number, to_timestamp, unbase64
from pyspark.sql.window import Window
import uuid

dbutils.widgets.text("TABLE_ID", "orders")
dbutils.widgets.text("CATALOG", "workspace")
dbutils.widgets.text("BRONZE_SCHEMA", "bronze")
dbutils.widgets.text("SILVER_SCHEMA", "silver")
dbutils.widgets.text("CHECKPOINT_ROOT", "/Volumes/workspace/default/mnt/checkpoints")
dbutils.widgets.text("MONITORING_SCHEMA", "monitoring")
dbutils.widgets.text("SCHEMA_POLICY", "additive_only")
dbutils.widgets.text("ALERT_CHANNEL", "log")
dbutils.widgets.text("WEBHOOK_URL", "")
dbutils.widgets.text("RUN_BOOTSTRAP_HOOKS", "false")


def parse_bool(raw_value: str) -> bool:
    return raw_value.strip().lower() in {"1", "true", "yes", "y"}


CONFIG = {
    "table_id": dbutils.widgets.get("TABLE_ID"),
    "catalog": dbutils.widgets.get("CATALOG"),
    "bronze_schema": dbutils.widgets.get("BRONZE_SCHEMA"),
    "silver_schema": dbutils.widgets.get("SILVER_SCHEMA"),
    "checkpoint_root": dbutils.widgets.get("CHECKPOINT_ROOT"),
    "monitoring_schema": dbutils.widgets.get("MONITORING_SCHEMA"),
    "schema_policy": dbutils.widgets.get("SCHEMA_POLICY"),
    "alert_channel": dbutils.widgets.get("ALERT_CHANNEL"),
    "webhook_url": dbutils.widgets.get("WEBHOOK_URL"),
    "run_bootstrap_hooks": parse_bool(dbutils.widgets.get("RUN_BOOTSTRAP_HOOKS")),
}

table_config = get_silver_table_config(CONFIG["table_id"])
PIPELINE_RUN_ID = str(uuid.uuid4())

bronze_table_fqn = f"{CONFIG['catalog']}.{CONFIG['bronze_schema']}.{table_config['bronze_table']}"
silver_table_fqn = f"{CONFIG['catalog']}.{CONFIG['silver_schema']}.{table_config['silver_table']}"
drift_table_fqn = f"{CONFIG['catalog']}.{CONFIG['monitoring_schema']}.schema_drift_log"
checkpoint_path = f"{CONFIG['checkpoint_root'].rstrip('/')}/{table_config['checkpoint_name']}"


In [ ]:
# ---------------------------------------------------------------------------
# Task 2.3 — Load PII config for this table
# Determines which columns to encrypt and what subject_id column to use.
# ---------------------------------------------------------------------------
import json as _pii_json

_PII_CONFIG_PATH = "/Volumes/workspace/default/mnt/pipeline_configs/pii/pii_config.json"

with open(_PII_CONFIG_PATH, "r") as _fh:
    _all_pii = _pii_json.load(_fh)

# Silver table short name, e.g. "silver.silver_customer"
_silver_short = f"{CONFIG['silver_schema']}.{table_config['silver_table']}"

# Columns that should be encrypted for this table
PII_ENCRYPT_COLS = [
    e for e in _all_pii
    if e["table"] == _silver_short and e["encrypt"] is True
]

# subject_id column (same for all encrypt=true entries of this table)
PII_SUBJECT_ID_COL  = PII_ENCRYPT_COLS[0]["subject_id_col"] if PII_ENCRYPT_COLS else None
PII_SUBJECT_TYPE    = _silver_short.split(".")[-1]           # e.g. "silver_customer"

if PII_ENCRYPT_COLS:
    print(f"PII encryption enabled for {_silver_short}: "
          f"columns={[e['column'] for e in PII_ENCRYPT_COLS]}, "
          f"subject_id_col={PII_SUBJECT_ID_COL}")
else:
    print(f"PII encryption disabled for {_silver_short} (no encrypt=true entries)")

In [ ]:
# ---------------------------------------------------------------------------
# Capture authentication context once in the driver as plain Python variables.
# Python closures will serialize these into the foreachBatch function.
# ---------------------------------------------------------------------------
import os as _os_creds

def get_databricks_config():
    """Capture workspace config for worker processes."""
    return {
        'host':  spark.conf.get("spark.databricks.workspaceUrl",
                                _os_creds.environ.get("DATABRICKS_HOST", "")),
        'token': dbutils.notebook.entry_point.getDbutils().notebook()
                        .getContext().apiToken().get(),
    }

# Capture once - these variables will be in closure scope
_db_host  = get_databricks_config()['host']
_db_token = get_databricks_config()['token']

print(f"Driver credentials captured: host={_db_host[:40]}...")

In [ ]:
create_schema_drift_table(spark, CONFIG['catalog'], CONFIG['monitoring_schema'])
ensure_schema_exists(CONFIG['catalog'], CONFIG['silver_schema'])
expected_schema = get_schema_contract(table_config['silver_contract_key'])
ensure_table_from_contract(spark, silver_table_fqn, expected_schema)

if CONFIG['run_bootstrap_hooks']:
    run_silver_bootstrap_hook(
        spark,
        table_config,
        silver_table_fqn,
        CONFIG['catalog'],
        CONFIG['silver_schema'],
    )

actual_schema = get_existing_schema(spark, silver_table_fqn)
print(f"Validating schema for {silver_table_fqn} with policy {CONFIG['schema_policy']}")

try:
    is_valid, drift_result = validate_schema_with_policy(
        spark,
        expected_schema=expected_schema,
        actual_schema=actual_schema,
        table_name=silver_table_fqn,
        drift_table=drift_table_fqn,
        source_system="postgres_cdc",
        pipeline_run_id=PIPELINE_RUN_ID,
        policy=CONFIG['schema_policy'],
        alert_channel=CONFIG['alert_channel'],
        webhook_url=CONFIG['webhook_url'] if CONFIG['webhook_url'] else None,
    )
    if drift_result['has_drift']:
        print(f"Schema drift detected: {drift_result['severity']}")
    else:
        print("Schema validation passed")
except SchemaDriftException as exc:
    dbutils.notebook.exit(f"Schema drift blocked pipeline: {exc}")


In [ ]:
# ---------------------------------------------------------------------------
# Parse Bronze stream → `parsed` DataFrame
#
# Build a generic Debezium envelope schema from this table's field_mappings:
# all before/after fields are read as StringType so that build_events_df
# can apply the right type transforms (epoch_micros, decimal_bytes, etc.).
# ---------------------------------------------------------------------------
from pyspark.sql.functions import from_json as _from_json
from pyspark.sql.types import StringType as _StringType, LongType as _LongType, StructType as _StructType, StructField as _StructField

# Collect every leaf field name referenced in source_paths (e.g. "rental_id" from "after.rental_id")
_leaf_fields: set[str] = set()
for _fm in table_config["field_mappings"]:
    for _path in _fm.get("source_paths", []):
        _parts = _path.split(".")
        if len(_parts) == 2:
            _leaf_fields.add(_parts[1])

_record_struct = _StructType([_StructField(f, _StringType(), True) for f in sorted(_leaf_fields)])

_envelope_schema = _StructType([
    _StructField("before", _record_struct,  True),
    _StructField("after",  _record_struct,  True),
    _StructField("op",     _StringType(),   True),
    _StructField("ts_ms",  _LongType(),     True),
])

_bronze_stream = (
    spark.readStream
    .option("mergeSchema", "true")
    .option("ignoreDeletes", "true")     # tolerate Bronze TRUNCATE without checkpoint reset
    .option("startingVersion", "0")      # on fresh checkpoint: read all Bronze history, not just latest
    .table(bronze_table_fqn)
)

parsed = (
    _bronze_stream
    .withColumnRenamed("offset",    "bronze_offset")
    .withColumnRenamed("partition", "bronze_partition")
    .withColumn("_env", _from_json(col("value"), _envelope_schema))
    .select(
        col("_env.before"),
        col("_env.after"),
        col("_env.op"),
        col("_env.ts_ms"),
        col("bronze_offset"),
        col("bronze_partition"),
        col("kafka_timestamp"),
        col("value").alias("bronze_value"),   # needed by decimal_from_json_paths transforms
    )
)

print(f"Bronze stream opened: {bronze_table_fqn}")
print(f"Envelope fields: {sorted(_leaf_fields)}")

In [ ]:
def coalesce_columns(source_paths: list[str]):
    columns = [col(path) for path in source_paths]
    if len(columns) == 1:
        return columns[0]
    return coalesce(*columns)


def coalesce_json_values(json_paths: list[str]):
    columns = [get_json_object(col('bronze_value'), json_path) for json_path in json_paths]
    if len(columns) == 1:
        return columns[0]
    return coalesce(*columns)


def build_events_df(parsed_df, field_mappings: list[dict]):
    working_df = parsed_df
    raw_columns_to_drop = []
    final_columns = []

    for field_mapping in field_mappings:
        target = field_mapping['target']
        transform = field_mapping.get('transform', 'plain')
        raw_column_name = f"__raw_{target}"

        if transform == 'plain':
            working_df = working_df.withColumn(target, coalesce_columns(field_mapping['source_paths']))
        elif transform == 'epoch_micros_to_timestamp':
            working_df = working_df.withColumn(raw_column_name, coalesce_columns(field_mapping['source_paths']))
            working_df = working_df.withColumn(target, to_timestamp((col(raw_column_name) / 1000000).cast('double')))
            raw_columns_to_drop.append(raw_column_name)
        elif transform == 'decimal_from_debezium_bytes':
            bytes_column_name = f"{raw_column_name}_bytes"
            int_column_name = f"{raw_column_name}_int"
            precision = field_mapping['precision']
            scale = field_mapping['scale']
            working_df = working_df.withColumn(raw_column_name, coalesce_columns(field_mapping['source_paths']))
            working_df = working_df.withColumn(bytes_column_name, unbase64(col(raw_column_name)))
            working_df = working_df.withColumn(int_column_name, expr(f"cast(conv(hex({bytes_column_name}), 16, 10) as double)"))
            working_df = working_df.withColumn(
                target,
                expr(f"cast({int_column_name} / pow(10, {scale}) as decimal({precision},{scale}))")
            )
            raw_columns_to_drop.extend([raw_column_name, bytes_column_name, int_column_name])
        elif transform == 'decimal_from_json_paths':
            bytes_column_name = f"{raw_column_name}_bytes"
            int_column_name = f"{raw_column_name}_int"
            scale_column_name = f"{raw_column_name}_scale"
            precision = field_mapping['precision']
            scale = field_mapping['scale']
            default_scale = field_mapping.get('default_scale', scale)
            working_df = working_df.withColumn(raw_column_name, coalesce_json_values(field_mapping['json_value_paths']))
            if field_mapping.get('json_scale_paths'):
                working_df = working_df.withColumn(scale_column_name, coalesce_json_values(field_mapping['json_scale_paths']).cast('int'))
                working_df = working_df.withColumn(scale_column_name, coalesce(col(scale_column_name), lit(default_scale)))
            else:
                working_df = working_df.withColumn(scale_column_name, lit(default_scale))
            working_df = working_df.withColumn(bytes_column_name, unbase64(col(raw_column_name)))
            working_df = working_df.withColumn(int_column_name, expr(f"cast(conv(hex({bytes_column_name}), 16, 10) as double)"))
            working_df = working_df.withColumn(
                target,
                expr(f"cast({int_column_name} / pow(10, {scale_column_name}) as decimal({precision},{scale}))")
            )
            raw_columns_to_drop.extend([raw_column_name, bytes_column_name, int_column_name, scale_column_name])
        else:
            raise ValueError(f"Unsupported transform '{transform}' for field '{target}'")

        final_columns.append(target)

    working_df = working_df.withColumn('event_ts_ms', col('ts_ms'))
    working_df = working_df.withColumn('event_time', to_timestamp((col('ts_ms') / 1000).cast('double')))
    final_columns.extend(['op', 'event_time', 'event_ts_ms', 'bronze_offset', 'bronze_partition', 'kafka_timestamp'])

    if raw_columns_to_drop:
        working_df = working_df.drop(*raw_columns_to_drop)

    return working_df.select(*final_columns)


events = build_events_df(parsed, table_config['field_mappings'])


In [ ]:
def upsert_to_silver(batch_df, batch_id):
    if not batch_df.take(1):
        return

    # Spark Connect: must use batch_df.sparkSession, NOT the global spark
    _spark = batch_df.sparkSession

    order_columns = [col(column_name).desc() for column_name in table_config['dedupe_order_columns']]
    window = Window.partitionBy(*table_config['primary_keys']).orderBy(*order_columns)
    latest = (
        batch_df
        .withColumn('__rn', row_number().over(window))
        .filter(col('__rn') == 1)
        .drop('__rn')
    )

    # ------------------------------------------------------------------ #
    # Task 2.3 — PII encryption                                           #
    # _db_host / _db_token are plain strings captured from driver scope.  #
    # cloudpickle serializes them into the foreachBatch closure.          #
    # ------------------------------------------------------------------ #
    if PII_ENCRYPT_COLS and PII_SUBJECT_ID_COL in latest.columns:
        from pyspark.sql.functions import udf as _pii_udf
        from pyspark.sql.types import BinaryType as _PiiBinaryType
        _get_dek_map   = get_dek_map
        _encrypt_value = encrypt_value

        # Set credentials in worker environment from closure variables
        import os as _os_worker
        _os_worker.environ['DATABRICKS_HOST']  = _db_host
        _os_worker.environ['DATABRICKS_TOKEN'] = _db_token

        subject_ids    = [r[0] for r in latest.select(PII_SUBJECT_ID_COL).distinct().collect()]
        dek_map        = _get_dek_map(subject_ids, PII_SUBJECT_TYPE)

        # Plain dict — cloudpickle-safe, no sparkContext needed
        _dek_map_local = dict(dek_map)

        def _encrypt_udf(value, subject_id):
            if value is None or subject_id is None:
                return None
            dek = _dek_map_local.get(str(subject_id))
            if dek is None:
                return None
            return _encrypt_value(str(value), dek)

        encrypt_udf = _pii_udf(_encrypt_udf, _PiiBinaryType())

        for entry in PII_ENCRYPT_COLS:
            col_name = entry["column"]
            if col_name in latest.columns:
                latest = latest.withColumn(
                    col_name,
                    encrypt_udf(col(col_name).cast("string"), col(PII_SUBJECT_ID_COL)),
                )

    # ------------------------------------------------------------------ #
    # MERGE into Silver                                                    #
    # ------------------------------------------------------------------ #
    existing_columns = get_existing_columns(_spark, silver_table_fqn)
    update_set, insert_values = build_merge_clauses(
        existing_columns,
        table_config['merge_core_fields'],
        include_inserted_dt=True,
        include_updated_dt=True,
    )

    join_condition = " AND ".join(
        f"t.{pk} = s.{pk}" for pk in table_config['primary_keys']
    )

    delta_table = DeltaTable.forName(_spark, silver_table_fqn)
    execute_merge(_spark, delta_table, latest, update_set, insert_values,
                  join_condition=join_condition)

    # ------------------------------------------------------------------ #
    # Task 2.4 — Post-MERGE DQ assertions                                 #
    # ------------------------------------------------------------------ #
    _run_silver_dq(_spark, batch_id)


def _run_silver_dq(_spark, batch_id: int) -> None:
    """Run post-MERGE DQ checks and write results to monitoring.dq_results."""
    pks       = table_config['primary_keys']
    tbl       = silver_table_fqn
    tbl_short = _silver_short
    run_id    = f"{PIPELINE_RUN_ID}:batch{batch_id}"

    pk_cols   = ", ".join(pks)
    dup_count = _spark.sql(f"""
        SELECT COUNT(*) - COUNT(DISTINCT {pk_cols}) AS dups FROM {tbl}
    """).collect()[0]["dups"]

    pk_status = "PASS" if dup_count == 0 else "FAIL"
    write_dq_result(
        _spark, run_id=run_id, layer="silver", table_name=tbl_short,
        check_name="pk_uniqueness", status=pk_status,
        observed_value=float(dup_count), threshold=0.0,
        message=f"{dup_count} duplicate PK(s) detected" if dup_count else None,
    )
    if pk_status == "FAIL":
        alert_dq_failure(
            table_name=tbl_short, check_name="pk_uniqueness",
            observed_value=dup_count, layer="silver",
            alert_channel=CONFIG["alert_channel"],
            webhook_url=CONFIG["webhook_url"] if CONFIG["webhook_url"] else None,
        )
        raise RuntimeError(f"DQ FAIL: PK uniqueness check failed for {tbl_short} — {dup_count} duplicate(s)")

    table_id = CONFIG["table_id"]

    if table_id == "rental":
        null_cust = _spark.sql(f"SELECT SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS n FROM {tbl}").collect()[0]["n"] or 0
        total     = _spark.sql(f"SELECT COUNT(*) AS n FROM {tbl}").collect()[0]["n"] or 1
        null_rate = null_cust / total
        write_dq_result(
            _spark, run_id=run_id, layer="silver", table_name=tbl_short,
            check_name="fk_customer_null_rate",
            status="WARN" if null_rate > 0 else "PASS",
            observed_value=null_rate, threshold=0.0,
            message=f"{null_cust} rows with null customer_id" if null_cust else None,
        )

    elif table_id == "payment":
        out_of_range = _spark.sql(f"""
            SELECT COUNT(*) AS n FROM {tbl}
            WHERE amount < 0.99 OR amount > 11.99
        """).collect()[0]["n"] or 0
        status = "FAIL" if out_of_range > 0 else "PASS"
        write_dq_result(
            _spark, run_id=run_id, layer="silver", table_name=tbl_short,
            check_name="payment_amount_range", status=status,
            observed_value=float(out_of_range), threshold=0.0,
            message=f"{out_of_range} payment(s) outside [0.99, 11.99]" if out_of_range else None,
        )
        if status == "FAIL":
            alert_dq_failure(
                table_name=tbl_short, check_name="payment_amount_range",
                observed_value=out_of_range, layer="silver",
                alert_channel=CONFIG["alert_channel"],
                webhook_url=CONFIG["webhook_url"] if CONFIG["webhook_url"] else None,
            )


query = (
    events.writeStream
    .foreachBatch(upsert_to_silver)
    .option('checkpointLocation', checkpoint_path)
    .option('mergeSchema', 'true')
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
print(f"Silver processing completed for {CONFIG['table_id']}. Run ID: {PIPELINE_RUN_ID}")